In [4]:
%pip install -q langchain wikipedia google-generativeai langchain-community langchain-google-genai python-dotenv


Note: you may need to restart the kernel to use updated packages.


# Connect the Gemini API


In [5]:
import os
import google.generativeai as genai

# Load the API key from a local .env file (git-ignored) or the shell environment, so this
# notebook runs from any IDE or terminal. In Google Colab, use Secrets instead:
#   from google.colab import userdata
#   GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
except ImportError:
    pass

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "YOUR_API_KEY_HERE")

# Configure the Gemini API
genai.configure(api_key=GOOGLE_API_KEY)


# Build a basic langchain agent


In [6]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

# Instantiate the Wikipedia tool
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000)
wikipedia_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

# Initialize the Gemini model (gemini-1.5-* is retired; use a current model)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, google_api_key=GOOGLE_API_KEY)

# Build a ReAct-style tool-calling agent.
# LangChain 1.x removed initialize_agent; create_agent (built on LangGraph) is the replacement.
agent = create_agent(llm, tools=[wikipedia_tool])


/var/folders/d3/nhglz3qs7sggp0t2xt84rjvh0000gn/T/ipykernel_30828/3931500495.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


# Run the agent


In [9]:
# Run the agent with a sample query
result = agent.invoke({"messages": [{"role": "user", "content": "What is the capital of Moon?"}]})

# Gemini returns the answer as content blocks; normalize the final message to plain text.
final = result["messages"][-1]
answer = final.content
if isinstance(answer, list):
    answer = "".join(b.get("text", "") for b in answer if isinstance(b, dict))
print(answer)


The Moon is not a country and therefore does not have a capital.
